In [12]:
import ee
import pandas as pd


# Solo la PRIMERA VEZ para autenticar
ee.Authenticate()

# Luego usa tu proyecto
PROJECT_ID = 'gen-lang-client-0247558984'  # Pega aquí tu Project ID

ee.Initialize(
    project=PROJECT_ID,
    opt_url='https://earthengine-highvolume.googleapis.com'
)

In [13]:

# 2. CONFIGURACIÓN
# Región delimitada en Google Earth (Arequipa, Perú)
roi = ee.Geometry.Polygon([[
    [-71.35625205956532, -16.437592546762467],
    [-71.33181177102162, -16.437592546762467],
    [-71.33181177102162, -16.42579946757347],
    [-71.35625205956532, -16.42579946757347],
    [-71.35625205956532, -16.437592546762467]
]], None, False)  # geodesic = False como en Google Earth

fecha_inicio = '2024-01-01'
fecha_fin = '2025-01-01'

# TODAS LAS BANDAS ESPECTRALES DISPONIBLES EN SENTINEL-2
bandas_espectrales = [
    'B1',   # Coastal aerosol (443 nm) - 60m
    'B2',   # Blue (490 nm) - 10m
    'B3',   # Green (560 nm) - 10m
    'B4',   # Red (665 nm) - 10m
    'B5',   # Red Edge 1 (705 nm) - 20m
    'B6',   # Red Edge 2 (740 nm) - 20m
    'B7',   # Red Edge 3 (783 nm) - 20m
    'B8',   # NIR (842 nm) - 10m
    'B8A',  # Narrow NIR (865 nm) - 20m
    'B9',   # Water vapour (945 nm) - 60m
    'B11',  # SWIR 1 (1610 nm) - 20m
    'B12'   # SWIR 2 (2190 nm) - 20m
]


In [14]:

# 3. FUNCIONES MEJORADAS
def mask_s2_clouds(image):
    """Máscara de nubes mejorada"""
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    
    # Añadimos SCL (Scene Classification) para filtrado adicional
    scl = image.select('SCL')
    # Eliminamos: saturated(1), dark(2), cloud_shadows(3), clouds(8,9), cirrus(10), snow(11)
    scl_mask = scl.gte(4).And(scl.lte(6))  # Mantener: vegetation(4), not_vegetated(5), water(6)
    
    combined_mask = mask.And(scl_mask)
    
    return image.updateMask(combined_mask).divide(10000).select(bandas_espectrales) \
                .copyProperties(image, ['system:time_start', 'CLOUDY_PIXEL_PERCENTAGE'])

def add_all_indices(image):
    """TODOS los índices de vegetación y espectrales relevantes para reforestación"""
    
    # ÍNDICES DE VEGETACIÓN
    # 1. NDVI - Normalized Difference Vegetation Index (salud vegetal general)
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    
    # 2. EVI - Enhanced Vegetation Index (mejor en zonas densas)
    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
            'NIR': image.select('B8'),
            'RED': image.select('B4'),
            'BLUE': image.select('B2')
        }).rename('EVI')
    
    # 3. SAVI - Soil Adjusted Vegetation Index (reduce efecto del suelo)
    savi = image.expression(
        '((NIR - RED) / (NIR + RED + 0.428)) * (1.428)', {
            'NIR': image.select('B8'),
            'RED': image.select('B4')
        }).rename('SAVI')
    
    # 4. NDRE - Normalized Difference Red Edge (clorofila, estrés vegetal)
    ndre = image.normalizedDifference(['B8', 'B5']).rename('NDRE')
    
    # 5. GNDVI - Green NDVI (contenido de clorofila)
    gndvi = image.normalizedDifference(['B8', 'B3']).rename('GNDVI')
    
    # 6. CIre - Chlorophyll Index Red Edge
    cire = image.expression(
        '(NIR / RE1) - 1', {
            'NIR': image.select('B8'),
            'RE1': image.select('B5')
        }).rename('CIre')
    
    # ÍNDICES DE AGUA/HUMEDAD
    # 7. NDWI - Normalized Difference Water Index (contenido de agua)
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    
    # 8. NDMI - Normalized Difference Moisture Index (humedad de la vegetación)
    ndmi = image.normalizedDifference(['B8', 'B11']).rename('NDMI')
    
    # 9. MSI - Moisture Stress Index
    msi = image.expression(
        'SWIR1 / NIR', {
            'SWIR1': image.select('B11'),
            'NIR': image.select('B8')
        }).rename('MSI')
    
    # ÍNDICES DE SUELO
    # 10. NBR - Normalized Burn Ratio (áreas quemadas, biomasa)
    nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR')
    
    # 11. BSI - Bare Soil Index (suelo desnudo)
    bsi = image.expression(
        '((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))', {
            'SWIR1': image.select('B11'),
            'RED': image.select('B4'),
            'NIR': image.select('B8'),
            'BLUE': image.select('B2')
        }).rename('BSI')
    
    # 12. NDBI - Normalized Difference Built-up Index (áreas construidas)
    ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')
    
    # ÍNDICES ADICIONALES ÚTILES PARA REFORESTACIÓN
    # 13. LAI - Leaf Area Index approximation
    lai = image.expression(
        '3.618 * EVI - 0.118', {
            'EVI': evi
        }).rename('LAI')
    
    # 14. ARVI - Atmospherically Resistant Vegetation Index
    arvi = image.expression(
        '(NIR - (2 * RED) + BLUE) / (NIR + (2 * RED) + BLUE)', {
            'NIR': image.select('B8'),
            'RED': image.select('B4'),
            'BLUE': image.select('B2')
        }).rename('ARVI')
    
    # 15. SIPI - Structure Insensitive Pigment Index
    sipi = image.expression(
        '(NIR - BLUE) / (NIR - RED)', {
            'NIR': image.select('B8'),
            'BLUE': image.select('B2'),
            'RED': image.select('B4')
        }).rename('SIPI')
    
    # 16. PSRI - Plant Senescence Reflectance Index (envejecimiento vegetal)
    psri = image.expression(
        '(RED - GREEN) / RE1', {
            'RED': image.select('B4'),
            'GREEN': image.select('B3'),
            'RE1': image.select('B6')
        }).rename('PSRI')
    
    return image.addBands([
        ndvi, evi, savi, ndre, gndvi, cire,
        ndwi, ndmi, msi,
        nbr, bsi, ndbi,
        lai, arvi, sipi, psri
    ])


In [15]:

# 4. PROCESAMIENTO
print("Cargando colección de imágenes Sentinel-2...")
collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(roi)
              .filterDate(fecha_inicio, fecha_fin)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))  # Más estricto
              .map(mask_s2_clouds)
              .map(add_all_indices))

# Contar imágenes disponibles
n_images = collection.size().getInfo()
print(f"Imágenes encontradas: {n_images}")


Cargando colección de imágenes Sentinel-2...
Imágenes encontradas: 127


In [16]:

# 5. EXTRACCIÓN DE DATOS
def reduce_region_function(image):
    """Extrae estadísticas de toda el área"""
    # Media de todos los valores
    stats_mean = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=10,
        maxPixels=1e9,
        bestEffort=True
    )
    
    # Desviación estándar (variabilidad espacial)
    stats_std = image.reduceRegion(
        reducer=ee.Reducer.stdDev(),
        geometry=roi,
        scale=10,
        maxPixels=1e9,
        bestEffort=True
    )
    
    # Valores mín y máx
    stats_minmax = image.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=roi,
        scale=10,
        maxPixels=1e9,
        bestEffort=True
    )
    
    # Combinamos todas las estadísticas
    all_stats = stats_mean.combine(stats_std).combine(stats_minmax)
    
    # Añadimos metadatos
    return ee.Feature(None, all_stats).set({
        'date': image.date().format('YYYY-MM-dd'),
        'timestamp': image.date().millis(),
        'cloud_percentage': image.get('CLOUDY_PIXEL_PERCENTAGE')
    })

# Convertimos la colección
print("Procesando datos...")
timeseries_data = collection.map(reduce_region_function)

# Filtrar imágenes válidas
timeseries_data = timeseries_data.filter(ee.Filter.notNull(['NDVI']))


Procesando datos...


In [17]:

# 6. DESCARGA DE DATOS
print("Descargando datos desde Google Earth Engine...")
data_json = timeseries_data.getInfo()

if not data_json['features']:
    print("⚠️ No se encontraron datos válidos para el período seleccionado")
else:
    # 7. CONVERSIÓN A PANDAS
    data_list = [item['properties'] for item in data_json['features']]
    df = pd.DataFrame(data_list)
    
    # Convertir fecha
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    
    print(f"\n✅ Datos descargados: {len(df)} observaciones")
    print(f"📊 Variables disponibles: {len(df.columns)}")
    print(f"\n🌿 BANDAS ESPECTRALES ({len(bandas_espectrales)}):")
    print(f"   {', '.join(bandas_espectrales)}")
    print(f"\n📈 ÍNDICES CALCULADOS (16):")
    indices = ['NDVI', 'EVI', 'SAVI', 'NDRE', 'GNDVI', 'CIre', 
               'NDWI', 'NDMI', 'MSI', 'NBR', 'BSI', 'NDBI', 
               'LAI', 'ARVI', 'SIPI', 'PSRI']
    print(f"   {', '.join(indices)}")
    
    print("\n📋 Primeras filas:")
    print(df.head())
    
    # GUARDAR DATOS
    output_file = 'datos_reforestacion_completos.csv'
    df.to_csv(output_file, index=False)
    print(f"\n💾 Datos guardados en: {output_file}")
    
    # ESTADÍSTICAS BÁSICAS
    print("\n📊 Estadísticas de NDVI:")
    print(df['NDVI'].describe())
    
    # ANÁLISIS RÁPIDO
    print(f"\n🔍 Análisis rápido:")
    print(f"   - NDVI promedio: {df['NDVI'].mean():.3f}")
    print(f"   - Humedad (NDMI) promedio: {df['NDMI'].mean():.3f}")
    print(f"   - Suelo desnudo (BSI) promedio: {df['BSI'].mean():.3f}")
    print(f"   - Cobertura de nubes promedio: {df['cloud_percentage'].mean():.1f}%")

Descargando datos desde Google Earth Engine...

✅ Datos descargados: 121 observaciones
📊 Variables disponibles: 87

🌿 BANDAS ESPECTRALES (12):
   B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11, B12

📈 ÍNDICES CALCULADOS (16):
   NDVI, EVI, SAVI, NDRE, GNDVI, CIre, NDWI, NDMI, MSI, NBR, BSI, NDBI, LAI, ARVI, SIPI, PSRI

📋 Primeras filas:
       ARVI  ARVI_max  ARVI_min        B1       B11  B11_max  B11_min  \
0  0.063093  0.547170 -0.150986  0.010136  0.033867   0.3386   0.1126   
1  0.062850  0.496727 -0.148055  0.010118  0.033952   0.3352   0.1088   
2  0.055634  0.529735 -0.112010  0.009795  0.033935   0.3521   0.1260   
3  0.055633  0.448615 -0.115826  0.009892  0.034079   0.3489   0.1241   
4  0.055516  0.468516 -0.109124  0.027327  0.036902   0.4107   0.1442   

        B12  B12_max  B12_min  ...  PSRI_min      SAVI  SAVI_max  SAVI_min  \
0  0.035714   0.3199   0.0692  ... -0.063556  0.038078  0.503239 -0.006544   
1  0.035775   0.3173   0.0701  ... -0.044840  0.037802  0.492733 -0.